# 1.4 不看上下文能怎麼猜？

# 第 1 章：從猜下一個字開始

前置：讀過入門指南的 Python 與 tensor 部分即可。這一章只用幾個字；不要先開完整 Transformer。每段 code 都可單獨執行。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：把分數換成比例：像分一塊餅**

一起加同常數不改比例，分數差才重要。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/softmax.svg")))

**先想一想：**資料中有十隻貓、一隻狗，這個方法會偏向誰？

還沒看前文時，最合理的簡單猜法是照常見程度抽字。這是基準，幫我們知道加入上下文是否有用。

**把直覺寫成數學：**$p(c)=n_c/\sum_j n_j$。機率是非負且加總為 1 的數。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from collections import Counter

counts = Counter("貓貓貓狗鳥")
probability = {c: n / sum(counts.values()) for c, n in counts.items()}
print(probability)
assert abs(sum(probability.values()) - 1) < 1e-6

**如何讀結果：**只知道整體頻率，無法區分「貓看」和「狗喫」後面需要不同字。

**只改一件事：**只增加狗的出現次數，重新估計分佈。
